In [1]:
import os, sys

os.environ["JAVA_HOME"]             = r"C:\Program Files\Java\jdk-19"
os.environ["PATH"]                  = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Lab2").master("local[*]").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Python:", sys.version)
print("PySpark:", spark.version)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PySpark: 3.5.1


# PHASE 1: Data Loading & Inspection


In [2]:
# Create Spark session and context
spark = SparkSession.builder \
    .appName("Lab2_Telco_Churn") \
    .master("local[*]") \
    .getOrCreate()
sc = spark.sparkContext

In [3]:
# Load dataset using sc.textFile()
raw_rdd = sc.textFile("dataset.csv")
raw_rdd.cache()

dataset.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [4]:
header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda line: line != header)

In [5]:
# Split rows into columns using map()
split_rdd = data_rdd.map(lambda line: line.split(",")).cache()

In [6]:
# Display sample records using take()
print("\nSample Records (first 3 rows):")
for record in split_rdd.take(3):
    print(record)


Sample Records (first 3 rows):
['7590-VHVEG', 'Female', '0', 'Yes', 'No', '1', 'No', 'No phone service', 'DSL', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Electronic check', '29.85', '29.85', 'No']
['5575-GNVDE', 'Male', '0', 'No', 'No', '34', 'Yes', 'No', 'DSL', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'One year', 'No', 'Mailed check', '56.95', '1889.5', 'No']
['3668-QPYBK', 'Male', '0', 'No', 'No', '2', 'Yes', 'No', 'DSL', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Mailed check', '53.85', '108.15', 'Yes']


In [7]:
# Count total records using count()
total_records = split_rdd.count()
print(f"\nTotal Records: {total_records}")


Total Records: 7043


In [8]:
# Count number of partitions using getNumPartitions()
num_partitions = split_rdd.getNumPartitions()
print(f"Number of Partitions: {num_partitions}")

print("\nColumn Names:")
print(header.split(","))

Number of Partitions: 2

Column Names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


# PHASE 2: Data Cleaning (RDD transformations only)

In [9]:
NUM_COLS = len(header.split(","))

# Step 1: Trim whitespace from all fields using map()
trimmed_rdd = split_rdd.map(lambda row: [field.strip() for field in row]).cache()

In [10]:
# Step 2: Fix malformed rows - keep only rows with correct number of columns using filter()
fixed_rdd = trimmed_rdd.filter(lambda row: len(row) == NUM_COLS).cache()
print(f"\nAfter fixing malformed rows: {fixed_rdd.count()} records")


After fixing malformed rows: 7043 records


In [11]:
# Step 3: Remove rows with missing/empty values using filter()
no_missing_rdd = fixed_rdd.filter(
    lambda row: all(field != "" and field.strip() != " " for field in row)
).cache()
print(f"After removing missing values: {no_missing_rdd.count()} records")

After removing missing values: 7032 records


In [12]:
# Step 4: Remove duplicate records using distinct()
# بدل distinct() عادي، نستخدم reduceByKey كبديل أخف
no_duplicates_rdd = no_missing_rdd \
    .map(lambda row: (row[0], row)) \
    .reduceByKey(lambda a, b: a) \
    .map(lambda kv: kv[1]) \
    .cache()

print(f"After removing duplicates: {no_duplicates_rdd.count()} records")

After removing duplicates: 7032 records


In [13]:
# Step 5: Convert numeric columns to proper types using map()
# Columns by index (0-based):
#   5  = tenure (int)
#   18 = MonthlyCharges (float)
#   19 = TotalCharges (float)

def convert_types(row):
    try:
        row[5]  = int(row[5])           # tenure
        row[18] = float(row[18])         # MonthlyCharges
        row[19] = float(row[19]) if row[19] not in ("", " ") else 0.0  # TotalCharges
        return row
    except (ValueError, IndexError):
        return None  # mark bad rows for removal

typed_rdd = no_duplicates_rdd.map(convert_types)

In [14]:
# Remove rows that failed type conversion
clean_rdd = typed_rdd.filter(lambda row: row is not None)
print(f"After type conversion & final cleanup: {clean_rdd.count()} records")

print("\nSample cleaned record:")
print(clean_rdd.take(1))

After type conversion & final cleanup: 7032 records

Sample cleaned record:
[['7590-VHVEG', 'Female', '0', 'Yes', 'No', 1, 'No', 'No phone service', 'DSL', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Electronic check', 29.85, 29.85, 'No']]


# PHASE 3: Core RDD Transformations

In [15]:
# --- 1. map() ---
# Purpose: Transform each row to extract (customerID, MonthlyCharges) tuple
customer_charges = clean_rdd.map(lambda row: (row[0], row[18]))
print("\n1. map() - Extract (customerID, MonthlyCharges):")
print(customer_charges.take(3))


1. map() - Extract (customerID, MonthlyCharges):
[('7590-VHVEG', 29.85), ('9237-HQITU', 70.7), ('1452-KIOVK', 89.1)]


In [16]:
# --- 2. filter() ---
# Purpose: Keep only customers with Fiber optic internet service (index 8 = InternetService)
fiber_customers = clean_rdd.filter(lambda row: row[8] == "Fiber optic")
print(f"\n2. filter() - Fiber optic customers count: {fiber_customers.count()}")
print(fiber_customers.take(2))


2. filter() - Fiber optic customers count: 3096
[['9237-HQITU', 'Female', '0', 'No', 'No', 2, 'Yes', 'No', 'Fiber optic', 'No', 'No', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Electronic check', 70.7, 151.65, 'Yes'], ['1452-KIOVK', 'Male', '0', 'No', 'Yes', 22, 'Yes', 'Yes', 'Fiber optic', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Month-to-month', 'Yes', 'Credit card (automatic)', 89.1, 1949.4, 'No']]


In [17]:
# --- 3. flatMap() ---
# Purpose: Extract all unique service features each customer has (Yes values)
# Columns 6-14 are service-related; get feature name for each "Yes"
service_cols = ["PhoneService", "MultipleLines", "InternetService",
                "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]

customer_services = clean_rdd.flatMap(
    lambda row: [(row[0], service_cols[i]) for i, col_idx in
                 enumerate(range(6, 15)) if row[col_idx] == "Yes"]
)
print(f"\n3. flatMap() - (customerID, service) pairs count: {customer_services.count()}")
print("Sample:", customer_services.take(4))


3. flatMap() - (customerID, service) pairs count: 23651
Sample: [('7590-VHVEG', 'OnlineBackup'), ('9237-HQITU', 'PhoneService'), ('1452-KIOVK', 'PhoneService'), ('1452-KIOVK', 'MultipleLines')]


In [18]:
# --- 4. distinct() ---
# Purpose: Get all unique contract types (index 16 = Contract)
contract_types = clean_rdd.map(lambda row: row[16]).distinct()
print(f"\n4. distinct() - Unique contract types: {contract_types.collect()}")


4. distinct() - Unique contract types: ['Yes', 'No']


In [19]:
# --- 5. sample() ---
# Purpose: Take a random 10% sample for exploratory analysis
sampled_rdd = clean_rdd.sample(withReplacement=False, fraction=0.1, seed=42)
print(f"\n5. sample() - 10% sample size: {sampled_rdd.count()} records")


5. sample() - 10% sample size: 688 records


In [20]:
# --- 6. mapPartitions() ---
# Purpose: Count rows per partition for load distribution analysis
def count_in_partition(iterator):
    count = sum(1 for _ in iterator)
    yield count

partition_counts = clean_rdd.mapPartitions(count_in_partition)
print(f"\n6. mapPartitions() - Records per partition: {partition_counts.collect()}")


6. mapPartitions() - Records per partition: [3606, 3426]


In [21]:
# --- 7. repartition() ---
# Purpose: Increase parallelism by repartitioning into 4 partitions
repartitioned_rdd = clean_rdd.repartition(4)
print(f"\n7. repartition() - New partition count: {repartitioned_rdd.getNumPartitions()}")


7. repartition() - New partition count: 4


In [22]:
# --- 8. coalesce() ---
# Purpose: Reduce partitions to 2 without full shuffle (efficient for writing output)
coalesced_rdd = clean_rdd.coalesce(2)
print(f"\n8. coalesce() - Coalesced partition count: {coalesced_rdd.getNumPartitions()}")


8. coalesce() - Coalesced partition count: 2


# PHASE 4: Key-Value Operations 

In [23]:
# Prepare key-value RDD: key = Contract type, value = MonthlyCharges
# Index 16 = Contract, Index 18 = MonthlyCharges
contract_charges_rdd = clean_rdd.map(lambda row: (row[16], row[18]))

# Index 20 = Churn, key-value for churn analysis
churn_rdd = clean_rdd.map(lambda row: (row[20], 1))

In [24]:
# --- 1. reduceByKey() ---
# Purpose: Sum MonthlyCharges per Contract type
total_charges_by_contract = contract_charges_rdd.reduceByKey(lambda a, b: a + b)
print("\n1. reduceByKey() - Total MonthlyCharges per Contract type:")
for item in total_charges_by_contract.collect():
    print(f"   {item[0]}: ${item[1]:,.2f}")


1. reduceByKey() - Total MonthlyCharges per Contract type:
   Yes: $306,658.65
   No: $149,002.35


In [25]:
# --- 2. groupByKey() ---
# Purpose: Group customers by gender (index 1 = gender)
gender_rdd = clean_rdd.map(lambda row: (row[1], row[0]))  # (gender, customerID)
grouped_by_gender = gender_rdd.groupByKey()
print("\n2. groupByKey() - Customer counts grouped by gender:")
for gender, customers in grouped_by_gender.collect():
    print(f"   {gender}: {len(list(customers))} customers")


2. groupByKey() - Customer counts grouped by gender:
   Female: 3483 customers
   Male: 3549 customers


In [26]:
# --- 3. countByKey() ---
# Purpose: Count number of customers per Churn status
churn_counts = churn_rdd.countByKey()
print("\n3. countByKey() - Customer count by Churn status:")
for status, count in churn_counts.items():
    print(f"   Churn={status}: {count} customers")


3. countByKey() - Customer count by Churn status:
   Churn=No: 5163 customers
   Churn=Yes: 1869 customers


In [27]:
# --- 4. sortByKey() ---
# Purpose: Sort contract types alphabetically
sorted_contracts = total_charges_by_contract.sortByKey(ascending=True)
print("\n4. sortByKey() - Contract types sorted alphabetically:")
for item in sorted_contracts.collect():
    print(f"   {item[0]}: ${item[1]:,.2f}")


4. sortByKey() - Contract types sorted alphabetically:
   No: $149,002.35
   Yes: $306,658.65


In [28]:
# --- 5. aggregateByKey() ---
# Purpose: Compute (min, max, sum, count) of MonthlyCharges per Contract type
# Zero value: (min=inf, max=-inf, sum=0, count=0)
zero_value = (float("inf"), float("-inf"), 0.0, 0)

def seq_func(acc, value):
    return (min(acc[0], value), max(acc[1], value), acc[2] + value, acc[3] + 1)

def comb_func(acc1, acc2):
    return (min(acc1[0], acc2[0]), max(acc1[1], acc2[1]),
            acc1[2] + acc2[2], acc1[3] + acc2[3])

agg_result = contract_charges_rdd.aggregateByKey(zero_value, seq_func, comb_func)
print("\n5. aggregateByKey() - MonthlyCharges stats (min, max, sum, count) per Contract:")
for contract, (mn, mx, sm, cnt) in agg_result.collect():
    avg = sm / cnt if cnt > 0 else 0
    print(f"   {contract}: min=${mn:.2f}, max=${mx:.2f}, avg=${avg:.2f}, count={cnt}")



5. aggregateByKey() - MonthlyCharges stats (min, max, sum, count) per Contract:
   Yes: min=$18.55, max=$118.75, avg=$73.57, count=4168
   No: min=$18.25, max=$118.60, avg=$52.03, count=2864


In [29]:
# --- 6. combineByKey() ---
# Purpose: Compute average TotalCharges per Internet service type
# Index 8 = InternetService, Index 19 = TotalCharges
internet_charges_rdd = clean_rdd.map(lambda row: (row[8], row[19]))

create_combiner = lambda v: (v, 1)
merge_value     = lambda acc, v: (acc[0] + v, acc[1] + 1)
merge_combiners = lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])

combined = internet_charges_rdd.combineByKey(create_combiner, merge_value, merge_combiners)
avg_charges_by_internet = combined.map(lambda kv: (kv[0], kv[1][0] / kv[1][1]))

print("\n6. combineByKey() - Avg TotalCharges per Internet Service type:")
for service, avg in avg_charges_by_internet.collect():
    print(f"   {service}: ${avg:,.2f}")


6. combineByKey() - Avg TotalCharges per Internet Service type:
   DSL: $2,119.79
   Fiber optic: $3,205.30
   No: $665.22


# PHASE 5: Partitioning & Performance

In [30]:
import time
# --- Analyze current partitions ---
print(f"\nInitial partition count (after cleaning): {clean_rdd.getNumPartitions()}")


Initial partition count (after cleaning): 2


In [31]:
# --- Execution WITHOUT caching ---
start = time.time()
_ = clean_rdd.filter(lambda r: r[20] == "Yes").count()
_ = clean_rdd.map(lambda r: r[18]).reduce(lambda a, b: a + b)
_ = clean_rdd.map(lambda r: (r[16], 1)).reduceByKey(lambda a, b: a + b).collect()
time_no_cache = time.time() - start
print(f"\nExecution time WITHOUT cache (3 actions): {time_no_cache:.4f} seconds")


Execution time WITHOUT cache (3 actions): 8.0331 seconds


In [32]:
# --- Cache the RDD ---
cached_rdd = clean_rdd.cache()
# Warm up the cache with one action
cached_rdd.count()

7032

In [33]:
# --- Execution WITH caching ---
start = time.time()
_ = cached_rdd.filter(lambda r: r[20] == "Yes").count()
_ = cached_rdd.map(lambda r: r[18]).reduce(lambda a, b: a + b)
_ = cached_rdd.map(lambda r: (r[16], 1)).reduceByKey(lambda a, b: a + b).collect()
time_with_cache = time.time() - start
print(f"Execution time WITH cache    (3 actions): {time_with_cache:.4f} seconds")

speedup = time_no_cache / time_with_cache if time_with_cache > 0 else float("inf")
print(f"Speedup factor: {speedup:.2f}x")

Execution time WITH cache    (3 actions): 7.9426 seconds
Speedup factor: 1.01x


In [34]:
# --- Repartition for parallelism ---
rdd_4p = clean_rdd.repartition(4)
print(f"\nAfter repartition(4): {rdd_4p.getNumPartitions()} partitions")

rdd_2p = rdd_4p.coalesce(2)
print(f"After coalesce(2):    {rdd_2p.getNumPartitions()} partitions")


After repartition(4): 4 partitions
After coalesce(2):    2 partitions


In [35]:
# --- Performance Discussion ---
print("""
Performance Observations:
-------------------------------------------------
1. Partitions:
   - Default partitions depend on HDFS block size or local parallelism.
   - repartition() triggers a full shuffle (expensive but balances data).
   - coalesce() merges partitions without a full shuffle (efficient for shrinking).

2. Caching (cache / persist):
   - cache() stores the RDD in memory after the first action.
   - Subsequent actions reuse the cached data, avoiding recomputation.
   - Speedup is most noticeable when the same RDD is used in multiple actions.
   - Use persist(StorageLevel.MEMORY_AND_DISK) if data doesn't fit in memory.

3. General Tips:
   - Avoid groupByKey() for aggregations — use reduceByKey() or aggregateByKey()
     instead, as they reduce data transferred across the network.
   - Keep partitions balanced: too few → underutilizes cores;
     too many → overhead from task scheduling.
""")

# Stop Spark context
sc.stop()
print("Spark context stopped. Lab 2 complete.")


Performance Observations:
-------------------------------------------------
1. Partitions:
   - Default partitions depend on HDFS block size or local parallelism.
   - repartition() triggers a full shuffle (expensive but balances data).
   - coalesce() merges partitions without a full shuffle (efficient for shrinking).

2. Caching (cache / persist):
   - cache() stores the RDD in memory after the first action.
   - Subsequent actions reuse the cached data, avoiding recomputation.
   - Speedup is most noticeable when the same RDD is used in multiple actions.
   - Use persist(StorageLevel.MEMORY_AND_DISK) if data doesn't fit in memory.

3. General Tips:
   - Avoid groupByKey() for aggregations — use reduceByKey() or aggregateByKey()
     instead, as they reduce data transferred across the network.
   - Keep partitions balanced: too few → underutilizes cores;
     too many → overhead from task scheduling.

Spark context stopped. Lab 2 complete.
